# DSEG660 – check on an NVIDIA T4 (Google Colab)
The reported results were produced on an Apple M3 Max GPU. This notebook reruns the pipeline's training and evaluation code on a T4 to show it reproduces there. It does not replace the reported runs.

1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run all** (about 15 minutes; set `FULL_RUN = True` in the last section for a full retrain, about 30 more minutes)

On the T4 the base model is loaded in fp16 with loss scaling, because the T4 has no native bf16. The Mac runs used bf16, so a few greedy replies can differ.

## 0. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!git clone -q https://github.com/rouaamohd/dseg660-lora-greeting.git /content/dseg660 || true
%cd /content/dseg660
!pip -q install transformers==5.24.1 peft==0.22.0 accelerate==1.16.0 tokenizers==0.23.3 safetensors==0.8.0
import torch, transformers, peft
print('torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__)
print(torch.cuda.get_device_name(0), '| compute capability', torch.cuda.get_device_capability(0))

## 1. Training check: first 10 steps of the final run
Same data (`data/train_revised.jsonl`), LoRA settings, learning rate and seed as the final adapter. Checks the CUDA path, the 16 GiB cap and the memory recording, and compares the step-10 loss with the Mac run.

In [ ]:
!python train.py --data data/train_revised.jsonl --out /content/t4_check --lr 2e-4 --epochs 0.0340136
import json
t4 = json.load(open('/content/t4_check/training_run.json'))
mac = json.load(open('adapters/final/training_run.json'))
print('device:', t4['device'], '| precision:', t4['precision'])
print(f"peak memory: {t4['peak_allocated_gib']} GiB allocated, {t4['peak_driver_allocated_gib']} GiB reserved (budget 16 GiB)")
print(f"step-10 loss: T4 {t4['log'][0]['loss']:.4f} | Mac {mac['log'][0]['loss']:.4f}")

## 2. Published final adapter on the 128-prompt evaluation set
Prompt only, standard chat template, non-thinking mode, greedy decoding, 64 new tokens, no system prompt; scored with the same `score.py` and manual decisions as the report.

In [ ]:
!python infer.py --data data/source/dseg660-evaluation.jsonl --adapter adapters/final --batch-size 1 --out runs_t4/final_benchmark.raw.jsonl
!python score.py runs_t4/final_benchmark.raw.jsonl data/source/dseg660-evaluation.jsonl --decisions runs/benchmark_decisions.json

In [ ]:
import json

TARGETS = {"first": ("Greeting first when expected", 80, 70), "keep": ("Greeting first on keep requests", 48, 42),
           "last": ("Greeting last on explicit opt-outs", 48, 37), "answer": ("Correct main answer", 128, 92),
           "joint": ("Placement and answer both correct", 128, 80)}


def counts(summary_path):
    s = json.load(open(summary_path))
    return {k: s["overall"][k] if k in ("answer", "joint") else s[k]["placement"] for k in TARGETS}, s["overall"]["pending"]


def compare(t4_prefix, mac_prefix="runs/final_benchmark", label="T4"):
    t4, pending = counts(t4_prefix + ".summary.json")
    mac, _ = counts(mac_prefix + ".summary.json")
    print(f"{'Target':38} {'Mac':>9} {label:>9} {'Needed':>8}  Met on " + label)
    for k, (name, n, need) in TARGETS.items():
        print(f"{name:38} {mac[k]:>4} /{n:<3} {t4[k]:>4} /{n:<3} {'>= ' + str(need):>8}  {'yes' if t4[k] >= need else 'NO'}")
    if pending:
        print(f"\n{pending} answer(s) need a manual check and are counted as wrong above; see {t4_prefix}.pending.jsonl")
    a = {json.loads(l)["id"]: json.loads(l)["reply"] for l in open(mac_prefix + ".raw.jsonl")}
    b = {json.loads(l)["id"]: json.loads(l)["reply"] for l in open(t4_prefix + ".raw.jsonl")}
    diff = [i for i in a if a[i] != b.get(i)]
    print(f"\nReplies identical to the Mac run: {len(a) - len(diff)} / {len(a)}")
    for i in diff[:10]:
        print(f"\n{i}\n  Mac: {a[i]!r}\n  {label}: {b.get(i)!r}")
    if len(diff) > 10:
        print(f"\n... {len(diff) - 10} more in {t4_prefix}.raw.jsonl")


compare("runs_t4/final_benchmark")

## 3. Optional: full retrain of the final adapter on the T4
All 294 steps with the same settings, then the same evaluation. Set `FULL_RUN = True` and run this cell.

In [ ]:
FULL_RUN = False
if FULL_RUN:
    !python train.py --data data/train_revised.jsonl --out /content/final_t4 --lr 2e-4 --epochs 1
    !python infer.py --data data/source/dseg660-evaluation.jsonl --adapter /content/final_t4 --batch-size 1 --out runs_t4/final_t4_benchmark.raw.jsonl
    !python score.py runs_t4/final_t4_benchmark.raw.jsonl data/source/dseg660-evaluation.jsonl --decisions runs/benchmark_decisions.json
    r = json.load(open('/content/final_t4/training_run.json'))
    print(f"training: {r['train_seconds_through_adapter_save']} s, peak {r['peak_allocated_gib']} GiB allocated, final loss {r['final_loss']}")
    compare('runs_t4/final_t4_benchmark', label='T4 retrain')